In [ ]:
import pandas as pd
import torch

# Per riproducibilità
torch.manual_seed(8347247)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df = pd.read_csv("../data/all_ships.csv")

In [ ]:
from sklearn.model_selection import train_test_split

X = df[["sex", "age", "age_missing", "class", "crew"]]
Y = df["survived"]

X_tensor = torch.tensor(X.values, dtype=torch.float32)
Y_tensor = torch.tensor(Y.values, dtype=torch.float32) # la BCEWithLogitsLoss richiede target float

# per riproducibilità si usa random_state fissato
X_train, X_test, Y_train, Y_test = train_test_split(X_tensor, Y_tensor, test_size=0.2, random_state=42, stratify=Y_tensor)

mean = X_train.mean(0)
std = X_train.std(0)

X_train_norm = (X_train - mean) / std
X_test_norm = (X_test - mean) / std

In [4]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train_norm, Y_train)
test_ds = TensorDataset(X_test_norm, Y_test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

In [ ]:
from torch import nn

class LogisticRegressor(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, out_features=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

In [6]:
class LogisticRegressorLogits(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features=1)

    def forward(self, x):
        return self.linear(x)

In [ ]:
from torch.optim import SGD
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import accuracy_score

def train_model(model, train_loader, test_loader, criterion, lr=0.05, epochs=300):
    writer = SummaryWriter(f'../results/{model._get_name()}')
    model = model.to(device)

    optimizer = SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=0.001)

    use_logits = isinstance(criterion, nn.BCEWithLogitsLoss)

    for epoch in range(epochs):
        model.train()

        train_loss = 0.0
        y_true = []
        y_pred = []
        
        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            output = model(X_batch)
            loss = criterion(output.view(-1), Y_batch) # output e Y_batch devono avere stesso shape (batch_size,)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            train_loss += loss.item() * X_batch.size(0)

            if use_logits:
                # l'output sono logits quindi li normalizzo a probabilità con la sigmoide
                probs = torch.sigmoid(output.view(-1))
            else:
                probs = output

            preds = (probs >= 0.5).int() 
            y_true.extend(Y_batch.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

        train_loss /= len(train_loader.dataset)
        train_acc = accuracy_score(y_true, y_pred)

        model.eval()

        test_loss = 0.0
        y_true = []
        y_pred = []

        with torch.no_grad():
            for X_batch, Y_batch in test_loader:
                X_batch = X_batch.to(device)
                Y_batch = Y_batch.to(device)

                output = model(X_batch)
                loss = criterion(output.view(-1), Y_batch) # output e Y_batch devono avere stesso shape (batch_size,)
                test_loss += loss.item() * X_batch.size(0)

                if use_logits:
                    # l'output sono logits quindi li normalizzo a probabilità con la sigmoide e uso la soglia per scegliere la classe
                    probs = torch.sigmoid(output.view(-1))
                else:
                    probs = output.view(-1)

                preds = (probs >= 0.5).int() 
                y_true.extend(Y_batch.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())

        test_loss /= len(test_loader.dataset)
        test_acc = accuracy_score(y_true, y_pred)

        writer.add_scalar('loss/train', train_loss, epoch)
        writer.add_scalar('accuracy/train', train_acc, epoch)
        writer.add_scalar('loss/test', test_loss, epoch)
        writer.add_scalar('accuracy/test', test_acc, epoch)

        if epoch % 50 == 0:
            print(f"Epoch {epoch+1}/{epochs} | train_loss {train_loss:.4f} | train_acc {train_acc:.4f} | test_loss {test_loss:.4f} | test_acc {test_acc:.4f}")

    print(f"Epoch {epoch+1}/{epochs} | train_loss {train_loss:.4f} | train_acc {train_acc:.4f} | test_loss {test_loss:.4f} | test_acc {test_acc:.4f}\n")
    writer.close()

    return model, train_loss, train_acc, test_loss, test_acc

In [8]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_model(model, criterion):
    model.eval()

    with torch.no_grad():
        output = model(X_test_norm.to(device))

        # se il modello restituisce logits applico sigmoid
        if isinstance(criterion, nn.BCEWithLogitsLoss):
            probs = torch.sigmoid(output.view(-1))
        else:
            probs = output.view(-1)

        y_pred = (probs >= 0.5).int()

    y_true = Y_test.cpu().numpy()
    y_pred = y_pred.cpu().numpy()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    return accuracy, precision, recall, f1

In [ ]:
base_model = LogisticRegressor(in_features=5)
criterion = nn.BCELoss()

base_model, base_train_loss, base_train_acc, base_test_loss, base_test_acc = train_model(base_model, train_loader, test_loader, criterion)
base_accuracy, base_precision, base_recall, base_f1 = evaluate_model(base_model, criterion)

logits_model = LogisticRegressorLogits(in_features=5)
counts = torch.bincount(Y_train.view(-1).to(torch.int64))
pos_weight = (counts[0] / counts[1]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

logits_model, logits_train_loss, logits_train_acc, logits_test_loss, logits_test_acc = train_model(logits_model, train_loader, test_loader, criterion)
logits_accuracy, logits_precision, logits_recall, logits_f1 = evaluate_model(logits_model, criterion)

Epoch 1/300 | train_loss 0.6199 | train_acc 0.6585 | test_loss 0.6050 | test_acc 0.6632
Epoch 51/300 | train_loss 0.6185 | train_acc 0.6624 | test_loss 0.6093 | test_acc 0.6736
Epoch 101/300 | train_loss 0.6173 | train_acc 0.6686 | test_loss 0.6119 | test_acc 0.6723
Epoch 151/300 | train_loss 0.6199 | train_acc 0.6650 | test_loss 0.6121 | test_acc 0.6658
Epoch 201/300 | train_loss 0.6190 | train_acc 0.6660 | test_loss 0.6040 | test_acc 0.6723
Epoch 251/300 | train_loss 0.6181 | train_acc 0.6611 | test_loss 0.6065 | test_acc 0.6840
Epoch 300/300 | train_loss 0.6187 | train_acc 0.6647 | test_loss 0.6155 | test_acc 0.6658
Epoch 1/300 | train_loss 0.9274 | train_acc 0.5599 | test_loss 0.9022 | test_acc 0.6086
Epoch 51/300 | train_loss 0.9303 | train_acc 0.5527 | test_loss 0.9273 | test_acc 0.5215
Epoch 101/300 | train_loss 0.9256 | train_acc 0.5544 | test_loss 0.9035 | test_acc 0.4993
Epoch 151/300 | train_loss 0.9242 | train_acc 0.5527 | test_loss 0.9055 | test_acc 0.5423
Epoch 201/300 | 

In [10]:
print("Model Comparison\n")

print("LogisticRegressor")
print(f"train_loss: {base_train_loss:.4f}")
print(f"train_acc: {base_train_acc:.4f}")
print(f"test_loss: {base_test_loss:.4f}")
print(f"test_acc: {base_test_acc:.4f}\n")

print(f"Accuracy: {base_accuracy:.4f}")
print(f"Precision: {base_precision:.4f}")
print(f"Recall: {base_recall:.4f}")
print(f"F1 Score: {base_f1:.4f}\n")

print("LogisticRegressorLogits")
print(f"train_loss: {logits_train_loss:.4f}")
print(f"train_acc: {logits_train_acc:.4f}")
print(f"test_loss: {logits_test_loss:.4f}")
print(f"test_acc: {logits_test_acc:.4f}\n")

print(f"Accuracy: {logits_accuracy:.4f}")
print(f"Precision: {logits_precision:.4f}")
print(f"Recall: {logits_recall:.4f}")
print(f"F1 Score: {logits_f1:.4f}")

Model Comparison

LogisticRegressor
train_loss: 0.6187
train_acc: 0.6647
test_loss: 0.6155
test_acc: 0.6658

Accuracy: 0.6658
Precision: 0.3793
Recall: 0.0440
F1 Score: 0.0789

LogisticRegressorLogits
train_loss: 0.9175
train_acc: 0.5472
test_loss: 0.8940
test_acc: 0.5462

Accuracy: 0.5462
Precision: 0.3945
Recall: 0.7400
F1 Score: 0.5146

